In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)

Seed: 42


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE3_DIR = (
    PROJECT_DIR /
    "phase3_classification"
)

P3A_DIR = (
    PHASE3_DIR /
    "phase3A_label_audit"
)

P3A_CONFIG_DIR = P3A_DIR / "configs"
P3A_TABLE_DIR = P3A_DIR / "tables"
P3A_PLOT_DIR = P3A_DIR / "plots"


for folder in [
    P3A_CONFIG_DIR,
    P3A_TABLE_DIR,
    P3A_PLOT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# Create later-phase folders now
for folder_name in [
    "phase3B_direct_baseline",
    "phase3C_classifier_development",
    "phase3D_final_evaluation"
]:

    (
        PHASE3_DIR /
        folder_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )


print("Phase 3 directory:")
print(PHASE3_DIR)

print("\n✓ Phase 3 structure ready.")

Phase 3 directory:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification

✓ Phase 3 structure ready.


In [ ]:
PHASE1F_DIR = (
    PROJECT_DIR /
    "phase1_segmentation" /
    "phase1F_final_model"
)

PHASE2_DIR = (
    PROJECT_DIR /
    "phase2_quantitative"
)


PHASE2_COMPLETE_PATH = (
    PHASE2_DIR /
    "PHASE2_COMPLETE.json"
)

PHASE2_FINAL_RECORD_PATH = (
    PHASE2_DIR /
    "PHASE2_FINAL_RECORD.json"
)


assert PHASE2_COMPLETE_PATH.exists()
assert PHASE2_FINAL_RECORD_PATH.exists()


with open(
    PHASE2_COMPLETE_PATH,
    "r"
) as f:
    phase2_completion = json.load(f)


with open(
    PHASE2_FINAL_RECORD_PATH,
    "r"
) as f:
    phase2_record = json.load(f)


assert (
    phase2_completion["status"]
    ==
    "COMPLETE"
)

assert (
    phase2_record["status"]
    ==
    "COMPLETE_AND_FROZEN"
)

assert (
    phase2_record["phase2_frozen"]
    is True
)


print("=" * 70)
print("UPSTREAM PIPELINE VERIFIED")
print("=" * 70)

print("\nPhase 1 segmentation : FROZEN")
print("Phase 2 measurements : FROZEN")

print(
    "\n✓ Phase 3 will consume upstream outputs "
    "without modifying them."
)

UPSTREAM PIPELINE VERIFIED

Phase 1 segmentation : FROZEN
Phase 2 measurements : FROZEN

✓ Phase 3 will consume upstream outputs without modifying them.


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)


with open(
    SPLIT_PATH,
    "r"
) as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)


print("Train      :", len(train_ids))
print("Validation :", len(val_ids))
print("Test       :", len(test_ids))

print("\n✓ Original patient-level split preserved.")

Train      : 400
Validation : 50
Test       : 50

✓ Original patient-level split preserved.


In [ ]:
# ================================================================
# PHASE 3A — CELL 6
# EXTRACT, LOCATE, AND VERIFY CAMUS DATASET
# ================================================================

import zipfile
from pathlib import Path


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


# ----------------------------------------------------------------
# EXTRACT DATASET ONLY IF IT IS NOT ALREADY AVAILABLE
# ----------------------------------------------------------------

if not DATASET_PATH.exists():

    # Make sure the original archive exists in Drive
    assert OUTER_ZIP_PATH.exists(), (
        f"Outer archive not found:\n{OUTER_ZIP_PATH}"
    )


    # ------------------------------------------------------------
    # STEP 1 — Extract outer archive
    # ------------------------------------------------------------

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    # Make sure inner CAMUS archive now exists
    assert INNER_ARCHIVE_PATH.exists(), (
        f"Inner CAMUS archive not found:\n"
        f"{INNER_ARCHIVE_PATH}"
    )


    # ------------------------------------------------------------
    # STEP 2 — Extract inner CAMUS archive
    # ------------------------------------------------------------

    print("Extracting CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )

    print("✓ CAMUS archive extracted.")


else:

    print("✓ CAMUS dataset already extracted.")


# ----------------------------------------------------------------
# VERIFY DATASET DIRECTORY
# ----------------------------------------------------------------

assert DATASET_PATH.exists(), (
    f"CAMUS dataset directory not found:\n{DATASET_PATH}"
)


print("\nDataset path:")
print(DATASET_PATH)


# ----------------------------------------------------------------
# LOCATE PATIENT DIRECTORIES
# ----------------------------------------------------------------

patient_dirs = sorted([
    path
    for path in DATASET_PATH.iterdir()
    if (
        path.is_dir()
        and path.name.startswith("patient")
    )
])


# ----------------------------------------------------------------
# VERIFY ALL 500 CAMUS PATIENTS
# ----------------------------------------------------------------

assert len(patient_dirs) == 500, (
    f"Expected 500 CAMUS patient directories, "
    f"but found {len(patient_dirs)}."
)


# Verify that every patient has the A4C information file
missing_info_files = [
    patient_dir.name
    for patient_dir in patient_dirs
    if not (
        patient_dir /
        "Info_4CH.cfg"
    ).exists()
]


assert len(missing_info_files) == 0, (
    "Some patients are missing Info_4CH.cfg:\n"
    + str(missing_info_files[:10])
)


# ----------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CAMUS DATASET READY")
print("=" * 70)

print("Patients found :", len(patient_dirs))
print("First patient  :", patient_dirs[0].name)
print("Last patient   :", patient_dirs[-1].name)

print(
    "Info_4CH files :",
    len(patient_dirs) - len(missing_info_files),
    "/",
    len(patient_dirs)
)

print("\n✓ Dataset extraction and verification complete.")
print("✓ patient_dirs is ready for Cell 7.")

✓ CAMUS dataset already extracted.

Dataset path:
/content/camus_data_final/database_nifti

CAMUS DATASET READY
Patients found : 500
First patient  : patient0001
Last patient   : patient0500
Info_4CH files : 500 / 500

✓ Dataset extraction and verification complete.
✓ patient_dirs is ready for Cell 7.


In [ ]:
def parse_info_cfg(cfg_path):

    info = {}

    with open(
        cfg_path,
        "r",
        encoding="utf-8"
    ) as f:

        for raw in f:

            line = raw.strip()

            if (
                not line
                or line.startswith("#")
                or ":" not in line
            ):
                continue

            key, value = line.split(
                ":",
                1
            )

            key = key.strip()
            value = value.strip()

            try:
                value = int(value)

            except ValueError:

                try:
                    value = float(value)

                except ValueError:
                    pass

            info[key] = value

    return info


metadata_records = []


split_lookup = {}

for pid in train_ids:
    split_lookup[pid] = "train"

for pid in val_ids:
    split_lookup[pid] = "validation"

for pid in test_ids:
    split_lookup[pid] = "test"


for patient_dir in patient_dirs:

    pid = patient_dir.name

    info = parse_info_cfg(
        patient_dir /
        "Info_4CH.cfg"
    )


    metadata_records.append({

        "patient_id":
            pid,

        "split":
            split_lookup[pid],

        "reference_EF_pct":
            float(info["EF"]),

        "age":
            info.get("Age"),

        "sex":
            info.get("Sex"),

        "image_quality":
            info.get("ImageQuality")
    })


metadata_df = pd.DataFrame(
    metadata_records
)


assert len(metadata_df) == 500
assert metadata_df["reference_EF_pct"].notna().all()


display(
    metadata_df.head()
)

print("\nPatients:", len(metadata_df))

print(
    "EF range:",
    metadata_df["reference_EF_pct"].min(),
    "to",
    metadata_df["reference_EF_pct"].max()
)

,patient_id,split,reference_EF_pct,age,sex,image_quality
0,patient0001,train,54.0,56,F,Good
1,patient0002,train,40.0,55,M,Medium
2,patient0003,train,51.0,36,F,Good
3,patient0004,test,56.0,79,F,Medium
4,patient0005,train,48.0,78,F,Good



Patients: 500
EF range: 4.0 to 81.0


In [ ]:
def three_band_EF_group(ef):

    if ef <= 40:
        return "EF_<=40"

    elif ef < 50:
        return "EF_41_49"

    else:
        return "EF_>=50"


def two_band_EF_group(ef):

    if ef < 50:
        return "EF_<50"

    else:
        return "EF_>=50"


metadata_df[
    "EF_group_3class"
] = (
    metadata_df[
        "reference_EF_pct"
    ].apply(
        three_band_EF_group
    )
)


metadata_df[
    "EF_group_2class"
] = (
    metadata_df[
        "reference_EF_pct"
    ].apply(
        two_band_EF_group
    )
)


display(
    metadata_df[
        [
            "patient_id",
            "split",
            "reference_EF_pct",
            "EF_group_3class",
            "EF_group_2class"
        ]
    ].head(10)
)

,patient_id,split,reference_EF_pct,EF_group_3class,EF_group_2class
0,patient0001,train,54.0,EF_>=50,EF_>=50
1,patient0002,train,40.0,EF_<=40,EF_<50
2,patient0003,train,51.0,EF_>=50,EF_>=50
3,patient0004,test,56.0,EF_>=50,EF_>=50
4,patient0005,train,48.0,EF_41_49,EF_<50
5,patient0006,train,49.0,EF_41_49,EF_<50
6,patient0007,train,41.0,EF_41_49,EF_<50
7,patient0008,train,29.0,EF_<=40,EF_<50
8,patient0009,train,25.0,EF_<=40,EF_<50
9,patient0010,train,51.0,EF_>=50,EF_>=50


In [ ]:
three_class_order = [
    "EF_<=40",
    "EF_41_49",
    "EF_>=50"
]


three_class_counts = (
    pd.crosstab(
        metadata_df["split"],
        metadata_df["EF_group_3class"]
    )
    .reindex(
        columns=three_class_order,
        fill_value=0
    )
)


three_class_counts[
    "Total"
] = (
    three_class_counts.sum(
        axis=1
    )
)


three_class_counts = (
    three_class_counts.reindex(
        [
            "train",
            "validation",
            "test"
        ]
    )
)


print("=" * 70)
print("THREE-CLASS EF DISTRIBUTION")
print("=" * 70)

display(
    three_class_counts
)


three_class_counts.to_csv(
    P3A_TABLE_DIR /
    "three_class_EF_distribution.csv"
)

THREE-CLASS EF DISTRIBUTION


EF_group_3class,EF_<=40,EF_41_49,EF_>=50,Total
split,,,,
train,143,125,132,400
validation,13,17,20,50
test,16,14,20,50


In [ ]:
two_class_order = [
    "EF_<50",
    "EF_>=50"
]


two_class_counts = (
    pd.crosstab(
        metadata_df["split"],
        metadata_df["EF_group_2class"]
    )
    .reindex(
        columns=two_class_order,
        fill_value=0
    )
)


two_class_counts[
    "Total"
] = (
    two_class_counts.sum(
        axis=1
    )
)


two_class_counts = (
    two_class_counts.reindex(
        [
            "train",
            "validation",
            "test"
        ]
    )
)


print("=" * 70)
print("TWO-CLASS EF DISTRIBUTION")
print("=" * 70)

display(
    two_class_counts
)


two_class_counts.to_csv(
    P3A_TABLE_DIR /
    "two_class_EF_distribution.csv"
)

TWO-CLASS EF DISTRIBUTION


EF_group_2class,EF_<50,EF_>=50,Total
split,,,
train,268,132,400
validation,30,20,50
test,30,20,50


In [ ]:
def distribution_percentages(
    count_table
):

    class_columns = [
        column
        for column in count_table.columns
        if column != "Total"
    ]


    percentages = (
        count_table[
            class_columns
        ]
        .div(
            count_table[
                "Total"
            ],
            axis=0
        )
        *
        100.0
    )


    return percentages


three_class_percentages = (
    distribution_percentages(
        three_class_counts
    )
)


two_class_percentages = (
    distribution_percentages(
        two_class_counts
    )
)


print("=" * 70)
print("THREE-CLASS DISTRIBUTION (%)")
print("=" * 70)

display(
    three_class_percentages.round(2)
)


print("\n" + "=" * 70)
print("TWO-CLASS DISTRIBUTION (%)")
print("=" * 70)

display(
    two_class_percentages.round(2)
)

THREE-CLASS DISTRIBUTION (%)


EF_group_3class,EF_<=40,EF_41_49,EF_>=50
split,,,
train,35.75,31.25,33.0
validation,26.00,34.00,40.0
test,32.00,28.00,40.0



TWO-CLASS DISTRIBUTION (%)


EF_group_2class,EF_<50,EF_>=50
split,,
train,67.0,33.0
validation,60.0,40.0
test,60.0,40.0


In [ ]:
split_summary = (

    metadata_df
    .groupby("split")[
        "reference_EF_pct"
    ]
    .agg([
        "count",
        "mean",
        "std",
        "median",
        "min",
        "max"
    ])
    .reindex([
        "train",
        "validation",
        "test"
    ])
)


display(
    split_summary.round(3)
)


split_summary.to_csv(
    P3A_TABLE_DIR /
    "reference_EF_split_summary.csv"
)

,count,mean,std,median,min,max
split,,,,,,
train,400,43.982,11.835,45.0,6.0,81.0
validation,50,45.540,13.123,47.5,4.0,69.0
test,50,44.860,12.352,47.0,13.0,68.0


In [ ]:
# ================================================================
# PHASE 3A — CELL 13
# FREEZE PRIMARY EF CLASSIFICATION TARGET
# ================================================================

PRIMARY_CLASS_ORDER = [
    "Lower_EF",
    "Intermediate_EF",
    "Higher_EF"
]


def reference_EF_to_class(ef):
    """
    Convert CAMUS reference EF into the frozen three-band
    functional classification used in Phase 3.

    These labels describe EF-based functional bands only.
    They are NOT heart-failure diagnoses.
    """

    ef = float(ef)

    if ef <= 40:
        return "Lower_EF"

    elif ef < 50:
        return "Intermediate_EF"

    else:
        return "Higher_EF"


metadata_df[
    "reference_EF_class"
] = (
    metadata_df[
        "reference_EF_pct"
    ].apply(
        reference_EF_to_class
    )
)


# ---------------------------------------------------------------
# VERIFY LABELS
# ---------------------------------------------------------------

assert (
    metadata_df[
        "reference_EF_class"
    ].notna().all()
)

assert set(
    metadata_df[
        "reference_EF_class"
    ].unique()
) == set(
    PRIMARY_CLASS_ORDER
)


# ---------------------------------------------------------------
# FINAL DISTRIBUTION
# ---------------------------------------------------------------

final_class_distribution = (
    pd.crosstab(
        metadata_df["split"],
        metadata_df["reference_EF_class"]
    )
    .reindex(
        index=[
            "train",
            "validation",
            "test"
        ],
        columns=PRIMARY_CLASS_ORDER,
        fill_value=0
    )
)


final_class_distribution[
    "Total"
] = (
    final_class_distribution.sum(
        axis=1
    )
)


print("=" * 72)
print("FROZEN PHASE 3 TARGET")
print("=" * 72)

print(
    "\nLower EF        : reference EF <= 40%"
)

print(
    "Intermediate EF : 40% < reference EF < 50%"
)

print(
    "Higher EF       : reference EF >= 50%"
)

print(
    "\nTerminology: EF-based functional bands; "
    "not heart-failure diagnoses."
)

print("\nFinal distribution:")

display(
    final_class_distribution
)

FROZEN PHASE 3 TARGET

Lower EF        : reference EF <= 40%
Intermediate EF : 40% < reference EF < 50%
Higher EF       : reference EF >= 50%

Terminology: EF-based functional bands; not heart-failure diagnoses.

Final distribution:


reference_EF_class,Lower_EF,Intermediate_EF,Higher_EF,Total
split,,,,
train,143,125,132,400
validation,13,17,20,50
test,16,14,20,50


In [ ]:
# ================================================================
# PHASE 3A — CELL 14
# SAVE FROZEN PATIENT-LEVEL LABEL TABLE
# ================================================================

phase3_label_df = (
    metadata_df[
        [
            "patient_id",
            "split",
            "reference_EF_pct",
            "reference_EF_class",
            "age",
            "sex",
            "image_quality"
        ]
    ]
    .sort_values(
        [
            "split",
            "patient_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


LABEL_TABLE_PATH = (
    P3A_TABLE_DIR /
    "phase3_reference_EF_labels.csv"
)


phase3_label_df.to_csv(
    LABEL_TABLE_PATH,
    index=False
)


# ---------------------------------------------------------------
# VERIFY
# ---------------------------------------------------------------

assert len(
    phase3_label_df
) == 500

assert (
    phase3_label_df[
        "reference_EF_class"
    ].notna().all()
)


print("=" * 72)
print("REFERENCE LABEL TABLE SAVED")
print("=" * 72)

display(
    phase3_label_df.head(10)
)

print("\nPatients:", len(phase3_label_df))

print("\nSaved:")
print(LABEL_TABLE_PATH)

REFERENCE LABEL TABLE SAVED


,patient_id,split,reference_EF_pct,reference_EF_class,age,sex,image_quality
0,patient0004,test,56.0,Higher_EF,79,F,Medium
1,patient0013,test,35.0,Lower_EF,71,M,Poor
2,patient0014,test,50.0,Higher_EF,79,F,Good
3,patient0016,test,50.0,Higher_EF,52,F,Good
4,patient0017,test,58.0,Higher_EF,73,F,Good
5,patient0045,test,38.0,Lower_EF,46,M,Good
6,patient0048,test,55.0,Higher_EF,71,M,Good
7,patient0053,test,21.0,Lower_EF,68,F,Medium
8,patient0058,test,31.0,Lower_EF,77,M,Good
9,patient0072,test,38.0,Lower_EF,61,M,Medium



Patients: 500

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3A_label_audit/tables/phase3_reference_EF_labels.csv


In [ ]:
# ================================================================
# PHASE 3A — CELL 15
# SAVE AND FREEZE CLASSIFICATION DESIGN
# ================================================================

PHASE3A_DECISION_PATH = (
    P3A_CONFIG_DIR /
    "phase3A_classification_target_decision.json"
)


phase3A_decision = {

    "phase":
        "Phase 3A — EF Classification Target Definition",

    "status":
        "COMPLETE_AND_FROZEN",

    "primary_task":
        "Three-class EF-based functional classification",

    "reference_source":
        "CAMUS Info_4CH.cfg EF",

    "target_definition": {

        "Lower_EF":
            "reference EF <= 40%",

        "Intermediate_EF":
            "40% < reference EF < 50%",

        "Higher_EF":
            "reference EF >= 50%"
    },

    "class_order": [
        "Lower_EF",
        "Intermediate_EF",
        "Higher_EF"
    ],

    "terminology_note":
        (
            "The categories are EF-based functional bands "
            "for an assistive, non-diagnostic research system. "
            "They are not used as standalone heart-failure "
            "diagnoses."
        ),

    "dataset": {
        "total_patients": 500,

        "train": {
            "n": 400,
            "Lower_EF": 143,
            "Intermediate_EF": 125,
            "Higher_EF": 132
        },

        "validation": {
            "n": 50,
            "Lower_EF": 13,
            "Intermediate_EF": 17,
            "Higher_EF": 20
        },

        "test": {
            "n": 50,
            "Lower_EF": 16,
            "Intermediate_EF": 14,
            "Higher_EF": 20
        }
    },

    "class_balance_decision":
        (
            "No resampling, SMOTE, or other synthetic balancing "
            "is introduced at target-definition stage because "
            "all three classes have substantial representation "
            "and the training distribution is approximately balanced."
        ),

    "split_policy":
        (
            "Original patient-level 400/50/50 split with seed 42 "
            "is retained."
        ),

    "test_policy":
        (
            "The test partition is not used for classifier "
            "selection, feature selection, hyperparameter selection, "
            "threshold adjustment, or calibration."
        ),

    "historical_test_note":
        (
            "The test partition had historical pilot exposure "
            "before the clean rebuild and is therefore not described "
            "as a pristine external test set."
        ),

    "target_frozen":
        True
}


with open(
    PHASE3A_DECISION_PATH,
    "w"
) as f:

    json.dump(
        phase3A_decision,
        f,
        indent=2
    )


print("=" * 72)
print("PHASE 3A COMPLETE")
print("=" * 72)

print("\nPrimary task:")
print("Three-class EF-based functional classification")

print("\nClasses:")
print("Lower EF        : <= 40%")
print("Intermediate EF : >40% and <50%")
print("Higher EF       : >= 50%")

print("\nClass balancing:")
print("No synthetic balancing introduced.")

print("\nTarget definition:")
print("FROZEN")

print("\nSaved:")
print(PHASE3A_DECISION_PATH)

PHASE 3A COMPLETE

Primary task:
Three-class EF-based functional classification

Classes:
Lower EF        : <= 40%
Intermediate EF : >40% and <50%
Higher EF       : >= 50%

Class balancing:
No synthetic balancing introduced.

Target definition:
FROZEN

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3A_label_audit/configs/phase3A_classification_target_decision.json
